# DispatchDesk RAG walkthrough

Use the application's configured chunking strategy, Jina embeddings, PostgreSQL/pgvector retrieval, and Groq answer service. The last cells demonstrate a conversation with a follow-up, using the same history format as Gradio.

Run `uv sync --locked` from `backend/` and select `backend/.venv/bin/python` as the notebook kernel. Run this notebook from the repository root, `backend/`, or `backend/notebooks/`.

Configure `DATABASE_URL`, `JINA_API_KEY`, and `GROQ_API_KEY` in the environment or `backend/.env`. Settings load credentials at runtime; no cell displays them. Apply migrations locally with `uv run alembic upgrade head` first.

The embedding cells call Jina; the storage cell upserts the document tables. Answer cells call Jina and Groq. Scenario data is neither changed nor included in chat.


In [ ]:
import sys
from pathlib import Path

import structlog


def find_backend_dir() -> Path:
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        for candidate in (directory, directory / "backend"):
            if (candidate / "service" / "rag.py").is_file() and (
                candidate / "config.py"
            ).is_file():
                return candidate
    raise FileNotFoundError("Run this notebook from within the DispatchDesk repository")


BACKEND_DIR = find_backend_dir()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from config import get_settings
from domain.chat import ChatMessage
from logging_config import configure_logging
from queries.vector_store import insert_chunks, retrieve
from service.corpus import CorpusService
from service.factory import (
    create_chunking_strategy,
    create_embedding_service,
    create_llm_service,
)
from service.rag import RAGService

configure_logging()
logger = structlog.stdlib.get_logger("dispatchdesk.notebook")
CORPUS_DIR = BACKEND_DIR / "service" / "rag_data" / "corpus"
settings = get_settings()
embedding_service = create_embedding_service(settings)
corpus_service = CorpusService(
    CORPUS_DIR,
    BACKEND_DIR,
    create_chunking_strategy(settings),
)
logger.info("Backend: %s; embedding model: %s", BACKEND_DIR, embedding_service.model)

## Load and chunk the corpus

Reuse `CorpusService` with the configured chunking strategy (Markdown sections by default). It excludes the corpus README and attaches document IDs, versions, source paths, section headings, and file hashes. Loading these files makes no API request or database write.


In [ ]:
documents, ids = corpus_service.load()
if not documents:
    raise ValueError("Corpus produced no chunks")

logger.info("Loaded %s section chunks", len(documents))
for chunk_id, document in list(zip(ids, documents, strict=True))[:3]:
    logger.info("%s | %s", chunk_id, document.metadata["section"])

## Embed document passages with Jina

This cell calls the Jina API. Documents and questions use the same model and input request payload as the original pipeline. The shared embedding client logs the API-reported token usage. There is no local embedding model or tokenizer download.


In [ ]:
texts = [document.page_content for document in documents]
embeddings = embedding_service.embed_documents(texts)

if len(embeddings) != len(documents):
    raise ValueError("The number of embeddings must match the number of chunks")
logger.info(
    "Generated %s embeddings with %s dimensions",
    len(embeddings),
    len(embeddings[0]),
)

## Store documents and chunks in PostgreSQL

Executing this cell writes to `app.documents` and `app.document_chunks` in the database configured by `DATABASE_URL`. It uses the existing SQLAlchemy query function, which upserts the batch in one transaction. Re-running unchanged corpus files updates their existing rows. Orders, riders, hourly metrics, and zones are not reset.


In [ ]:
stored = insert_chunks(documents=documents, embeddings=embeddings)
logger.info("Upserted %s chunks into app.documents / app.document_chunks", stored)

## Retrieve relevant guidance

Embed only the question, then use the shared PostgreSQL cosine-distance query. Retrieved passages include the chunk citation, content, and similarity. A nearest-neighbor result is a candidate source; its score alone does not prove it answers the question.


In [ ]:
question = (
    "Packed orders are piling up. Can I batch frozen-item orders with other deliveries?"
)
TOP_K = 3
query_embedding = embedding_service.embed_query(question)
results = retrieve(query_embedding=query_embedding, match_count=TOP_K)

if not results:
    logger.info("No guidance is stored. Run the ingestion cells first.")
for result in results:
    logger.info(
        "Source: %s | similarity: %.3f",
        result["chunk_id"],
        result["similarity"],
    )
    logger.info("%s", result["content"])

## Generate a grounded answer

Compose `RAGService` with the same factories and settings as Gradio. It loads the existing system prompt and retrieves its own evidence. This cell starts a new conversation; rerun it to reset the notebook history.


In [ ]:
rag_service = RAGService(
    embedding_service=embedding_service,
    llm_service=create_llm_service(settings),
)
history: list[ChatMessage] = []
answer = rag_service.answer_question(question, top_k=TOP_K, history=history)
history.extend(
    [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ],
)
logger.info("%s", answer)

## Continue the conversation

A short follow-up uses the recent exchange to retrieve relevant passages and sends the full session history to Groq. This matches the UI pipeline without adding operational database rows to the prompt. Rerunning this cell continues the conversation; rerun the previous cell to start over.


In [ ]:
follow_up = "Why? What should I do instead?"
follow_up_answer = rag_service.answer_question(follow_up, top_k=TOP_K, history=history)
history.extend(
    [
        {"role": "user", "content": follow_up},
        {"role": "assistant", "content": follow_up_answer},
    ],
)
logger.info("%s", follow_up_answer)